# Logistic-Regression Surrogate Explanations → CSV

This tutorial trains XGBoost and MLP models on the CoXAM `wine_quality` dataset,
fits a **logistic-regression surrogate** (`LogisticRegressionSurrogateMethod`) on each
model’s predictions, generates per-instance explanations, and saves them to
`tutorials/generated_explanation/` in the standard project CSV formats.

**What a logistic-regression surrogate is**

A surrogate is a simpler, interpretable model trained to mimic the black-box AI model.
Here, a logistic regression is fitted on the AI model’s *predictions* (not ground-truth
labels).  For each test instance the explanation is the vector of per-feature
contributions `coefficient × feature_value`, making it a *weights*-style explanation.

**Covered method**

| Registry key | Class | Description |
|---|---|---|
| `logistic_regression` / `lr` / `weights` | `LogisticRegressionSurrogateMethod` | Per-feature coefficient × value contributions |

**Output files** (saved to `tutorials/generated_explanation/`)

| File | Schema | Description |
|---|---|---|
| `lr_xgboost_wine_quality.csv` | Attribution | Per-instance feature contributions — XGBoost surrogate |
| `lr_mlp_wine_quality.csv` | Attribution | Per-instance feature contributions — MLP surrogate |
| `lr_xgboost_wine_quality_table.csv` | CoXAM surrogate | Global LR coefficients — XGBoost surrogate |
| `lr_mlp_wine_quality_table.csv` | CoXAM surrogate | Global LR coefficients — MLP surrogate |

## 1 · Setup

> **Important — run this cell first.**  
> `OMP_NUM_THREADS=1` and `MKL_NUM_THREADS=1` must be set **before** any import.
> On macOS, PyTorch’s OpenMP runtime can conflict with the Jupyter kernel’s fork model
> and cause a silent crash without these.

In [6]:
import os, io, contextlib
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

from pathlib import Path
import sys

repo_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src").exists() and (candidate / "assets").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.ai_models import MLPUnifiedModel, XGBoostUnifiedModel
from src.xai_adapter import LogisticRegressionSurrogateMethod

OUTPUT_DIR = repo_root / "tutorials" / "generated_explanation"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"repo_root:  {repo_root}")
print(f"output_dir: {OUTPUT_DIR}")

repo_root:  /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api
output_dir: /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/generated_explanation


## 2 · Load CoXAM Data

The CoXAM assets live in `assets/ai_dataset/coxam/`.  We use this source (400 instances)
rather than CoAX (120 instances) for more stable model training and surrogate fitting.

The CoXAM values table has columns `x0`–`x19` to accommodate different datasets.
For `wine_quality` only `x0`–`x5` carry data; the rest are `NaN`.  We filter to
non-`NaN` columns using the metadata row.

In [7]:
ASSETS     = repo_root / "assets" / "ai_dataset" / "coxam"
DATASET_ID = "wine_quality"

values_df   = pd.read_csv(ASSETS / "values.csv")
metadata_df = pd.read_csv(ASSETS / "metadata.csv")

data = values_df[values_df["dataId"] == DATASET_ID].reset_index(drop=True)
meta = metadata_df[metadata_df["dataId"] == DATASET_ID].iloc[0]

all_xcols     = [c for c in data.columns if c.startswith("x") and c != "y"]
feature_cols  = [c for c in all_xcols if pd.notna(meta.get(c))]
feature_names = [str(meta[c]) for c in feature_cols]

X            = data[feature_cols].values.astype(float)
y            = data["y"].values.astype(int)
instance_ids = data["instanceId"].values

print(f"Dataset : {DATASET_ID}")
print(f"Shape   : {X.shape}")
print(f"Features: {list(zip(feature_cols, feature_names))}")
print(f"Classes : {dict(zip(*np.unique(y, return_counts=True)))}")
data[feature_cols + ["y", "instanceId"]].head()

Dataset : wine_quality
Shape   : (400, 6)
Features: [('x0', 'Alcohol'), ('x1', 'Sulphates'), ('x2', 'SO2'), ('x3', 'Vinegar Taint'), ('x4', 'pH'), ('x5', 'Chlorides')]
Classes : {np.int64(0): np.int64(201), np.int64(1): np.int64(199)}


,x0,x1,x2,x3,x4,x5,y,instanceId
0,10.8,0.63,30.0,0.39,3.18,0.093,1.0,0
1,9.5,0.62,35.0,0.52,3.38,0.085,1.0,1
2,8.7,0.75,55.0,0.42,3.34,0.084,0.0,2
3,10.2,0.87,15.0,0.46,2.99,0.071,0.0,3
4,9.5,0.62,78.0,0.45,3.38,0.080,0.0,4


## 3 · Train AI Models

We train two built-in AI models on the same split:

- **XGBoost** (`XGBoostUnifiedModel`) — gradient-boosted trees, 50 boosting rounds.
- **MLP** (`MLPUnifiedModel`) — two-layer PyTorch perceptron, 300 epochs, batch size 32.

Both use `cognitive_agent='coxam'`.  Their **predictions** (not ground-truth labels)
are what the LR surrogate will approximate.

> **Kernel-safe training**
> - XGBoost is trained **without a dev set** to prevent its C++ library from writing
>   verbose logs directly to the kernel’s stdout pipe (which crashes the kernel).
> - MLP epoch output is captured with `contextlib.redirect_stdout` and suppressed.
> - In both cases accuracy is measured separately with `.evaluate()`.

In [8]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, instance_ids, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples   Test: {X_test.shape[0]} samples")

Train: 320 samples   Test: 80 samples


In [9]:
xgb_model = XGBoostUnifiedModel(dataset_name=DATASET_ID, cognitive_agent="coxam")

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    xgb_model.train(X_train, y_train)

xgb_acc = xgb_model.evaluate(X_test, y_test)
print(f"XGBoost trained ({buf.getvalue().count(chr(10))} log lines)")
print(f"XGBoost test accuracy: {xgb_acc:.3f}")

XGBoost trained (50 log lines)
XGBoost test accuracy: 0.775


In [10]:
mlp_model = MLPUnifiedModel(
    dataset_name=DATASET_ID,
    input_dim=X_train.shape[1],
    num_classes=2,
    cognitive_agent="coxam",
)

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    mlp_model.train(X_train, y_train, epochs=300, batch_size=32)

mlp_acc = mlp_model.evaluate(X_test, y_test)
print(f"MLP trained ({buf.getvalue().count('Epoch')} epochs, output suppressed)")
print(f"MLP test accuracy: {mlp_acc:.3f}")

MLP trained (300 epochs, output suppressed)
MLP test accuracy: 0.812


## 4 · Generate Model Predictions for Surrogate Fitting

A surrogate approximates the **AI model’s behaviour**, not the ground-truth labels.
We call `.predict()` on the training set and take `argmax` to convert probability
arrays to class indices.  These become the `y` that the LR surrogate trains on.

`model.predict(X)` returns an `(n_samples, n_classes)` probability array.

In [11]:
y_xgb = np.argmax(xgb_model.predict(X_train), axis=1)
y_mlp = np.argmax(mlp_model.predict(X_train), axis=1)

print(f"XGBoost train label dist: {dict(zip(*np.unique(y_xgb, return_counts=True)))}")
print(f"MLP     train label dist: {dict(zip(*np.unique(y_mlp, return_counts=True)))}")

XGBoost train label dist: {np.int64(0): np.int64(156), np.int64(1): np.int64(164)}
MLP     train label dist: {np.int64(0): np.int64(142), np.int64(1): np.int64(178)}


## 5 · Fit Logistic-Regression Surrogate

`lr.fit(X_train, y_model)` fits an L2-regularized logistic regression to the model
predictions.  Internally the surrogate:
1. Min-max normalizes the features (CoXAM convention)
2. Fits `sklearn.linear_model.LogisticRegression` with regularization `C`
3. Converts normalized coefficients back to raw-feature space
4. Computes **fidelity** — fraction where surrogate agrees with the AI model

Key constructor parameters:
- `app_id`        — written to the `dataId` column in output CSVs
- `model_name`    — written to the `modelName` column in output CSVs
- `C`             — inverse regularization strength (larger = less regularization)
- `max_iter`      — max solver iterations
- `feature_names` — human-readable names stored in the metadata table

In [12]:
lr_xgb = LogisticRegressionSurrogateMethod(
    app_id=DATASET_ID,
    model_name="xgboost",
    C=1.0,
    max_iter=1000,
    feature_names=feature_names,
)
lr_xgb.fit(X_train, y_xgb)

print(f"LR surrogate (XGBoost)  fidelity    : {lr_xgb.fidelity:.3f}")
print(f"                        intercept   : {lr_xgb.get_intercept():.4f}")
print(f"                        coefficients: {lr_xgb.get_coefficients()}")

LR surrogate (XGBoost)  fidelity    : 0.863
                        intercept   : -3.3377
                        coefficients: {'a0': np.float64(0.7851081473458238), 'a1': np.float64(2.4397790127806593), 'a2': np.float64(-0.005324560017529655), 'a3': np.float64(-2.351080738891233), 'a4': np.float64(-1.5840819756704971), 'a5': np.float64(-2.840662243453125)}


In [13]:
lr_mlp = LogisticRegressionSurrogateMethod(
    app_id=DATASET_ID,
    model_name="mlp",
    C=1.0,
    max_iter=1000,
    feature_names=feature_names,
)
lr_mlp.fit(X_train, y_mlp)

print(f"LR surrogate (MLP)      fidelity    : {lr_mlp.fidelity:.3f}")
print(f"                        intercept   : {lr_mlp.get_intercept():.4f}")
print(f"                        coefficients: {lr_mlp.get_coefficients()}")

LR surrogate (MLP)      fidelity    : 0.950
                        intercept   : -1.7757
                        coefficients: {'a0': np.float64(0.9397658097470775), 'a1': np.float64(3.376042812185472), 'a2': np.float64(-0.0068709536270126085), 'a3': np.float64(-2.722750715855618), 'a4': np.float64(-2.5873416737052475), 'a5': np.float64(-3.452068948005095)}


## 6 · Compare Fidelity and Coefficients

Each coefficient shows the direction of influence on the positive class (`y=1`).
A positive coefficient means that feature increases the predicted probability;
a negative one decreases it.  Comparing coefficients between models reveals which
features each model relies on most.

In [14]:
coef_xgb = lr_xgb.get_coefficients()
coef_mlp = lr_mlp.get_coefficients()

all_keys = sorted(set(coef_xgb) | set(coef_mlp),
                  key=lambda k: int(k.split("=")[0][1:]))

coef_df = pd.DataFrame({
    "feature"     : [feature_names[int(k[1:])] if "=" not in k else k for k in all_keys],
    "coef_xgboost": [coef_xgb.get(k, float("nan")) for k in all_keys],
    "coef_mlp"    : [coef_mlp.get(k, float("nan")) for k in all_keys],
})

pd.concat([
    coef_df,
    pd.DataFrame([{"feature": "\u2014 fidelity \u2014",
                   "coef_xgboost": lr_xgb.fidelity,
                   "coef_mlp": lr_mlp.fidelity}]),
], ignore_index=True)

,feature,coef_xgboost,coef_mlp
0,Alcohol,0.785108,0.939766
1,Sulphates,2.439779,3.376043
2,SO2,-0.005325,-0.006871
3,Vinegar Taint,-2.351081,-2.722751
4,pH,-1.584082,-2.587342
5,Chlorides,-2.840662,-3.452069
6,— fidelity —,0.862500,0.950000


## 7 · Explain Test Instances

`lr.explain(X_test)` computes per-feature contributions as `coefficient × feature_value`
for each test instance.  The result is an `XAIAdapterResult` with:

- `result.values`      — `(n_instances, n_features)` contribution matrix
- `result.base_values` — `(n_instances,)` filled with the LR intercept value
- `result.method`      — `'logistic_regression'`
- `result.metadata`    — fidelity, app_id, model_name, per-instance probabilities, coefficients

In [15]:
preds_xgb_test = np.argmax(xgb_model.predict(X_test), axis=1)
preds_mlp_test = np.argmax(mlp_model.predict(X_test), axis=1)

result_lr_xgb = lr_xgb.explain(X_test)
result_lr_mlp = lr_mlp.explain(X_test)

print(f"LR/XGBoost  shape={result_lr_xgb.values.shape}  method='{result_lr_xgb.method}'")
print(f"LR/MLP      shape={result_lr_mlp.values.shape}  method='{result_lr_mlp.method}'")

LR/XGBoost  shape=(80, 6)  method='logistic_regression'
LR/MLP      shape=(80, 6)  method='logistic_regression'


In [16]:
df_lr_xgb = result_lr_xgb.to_explanation_df(ids_test, preds_xgb_test, DATASET_ID, "xgboost")
print(f"Shape: {df_lr_xgb.shape}  columns: {df_lr_xgb.columns.tolist()}")
df_lr_xgb.head(3)

Shape: (80, 13)  columns: ['dataId', 'modelName', 'expMethod', 'instanceId', 'pred', 'i_max', 'a0_i', 'a1_i', 'a2_i', 'a3_i', 'a4_i', 'a5_i', 'intercept']


,dataId,modelName,expMethod,instanceId,pred,i_max,a0_i,a1_i,a2_i,a3_i,a4_i,a5_i,intercept
0,wine_quality,xgboost,logistic_regression,209,0,7.065973,1.0,0.203719,-0.026374,-0.181339,-0.739809,-0.032162,-3.337674
1,wine_quality,xgboost,logistic_regression,280,0,7.144484,1.0,0.211725,-0.046952,-0.190864,-0.702855,-0.038170,-3.337674
2,wine_quality,xgboost,logistic_regression,33,0,9.028744,1.0,0.148623,-0.008256,-0.153636,-0.600035,-0.028631,-3.337674


## 8 · Save Per-Instance Explanation CSV

Schema: `dataId, modelName, expMethod, instanceId, pred, i_max, a0_i…aN_i, intercept`
- `aN_i`     — contribution for feature N, normalized by `i_max`
- `i_max`    — max absolute contribution for this instance
- `intercept`— LR intercept value (same for all instances of a given surrogate)

In [17]:
xgb_attr_path = OUTPUT_DIR / f"lr_xgboost_{DATASET_ID}.csv"
mlp_attr_path = OUTPUT_DIR / f"lr_mlp_{DATASET_ID}.csv"

result_lr_xgb.save_csv(xgb_attr_path, ids_test, preds_xgb_test, DATASET_ID, "xgboost")
result_lr_mlp.save_csv(mlp_attr_path, ids_test, preds_mlp_test, DATASET_ID, "mlp")

print(f"Saved \u2192 {xgb_attr_path.relative_to(repo_root)}")
print(f"Saved \u2192 {mlp_attr_path.relative_to(repo_root)}")

Saved → tutorials/generated_explanation/lr_xgboost_wine_quality.csv
Saved → tutorials/generated_explanation/lr_mlp_wine_quality.csv


## 9 · Save Surrogate Model Tables

`lr.to_explanation_table()` returns the CoXAM-style surrogate table — one row per
fitted model — containing the intercept and all coefficients (`coef_v0`, `coef_v1`, …)
plus fidelity.  Matches the schema of `assets/explanations/coxam/logistic_regression.csv`.

`lr.to_metadata_table()` returns feature-level metadata (names, min/max bounds).

In [18]:
xgb_table_path = OUTPUT_DIR / f"lr_xgboost_{DATASET_ID}_table.csv"
mlp_table_path = OUTPUT_DIR / f"lr_mlp_{DATASET_ID}_table.csv"

lr_xgb.to_explanation_table().to_csv(xgb_table_path, index=False)
lr_xgb.to_metadata_table().to_csv(
    OUTPUT_DIR / f"lr_xgboost_{DATASET_ID}_metadata.csv", index=False
)
lr_mlp.to_explanation_table().to_csv(mlp_table_path, index=False)
lr_mlp.to_metadata_table().to_csv(
    OUTPUT_DIR / f"lr_mlp_{DATASET_ID}_metadata.csv", index=False
)

print(f"Saved \u2192 {xgb_table_path.relative_to(repo_root)}")
print(f"Saved \u2192 {mlp_table_path.relative_to(repo_root)}")
print(f"\nSurrogate table columns: {lr_xgb.to_explanation_table().columns.tolist()}")

Saved → tutorials/generated_explanation/lr_xgboost_wine_quality_table.csv
Saved → tutorials/generated_explanation/lr_mlp_wine_quality_table.csv

Surrogate table columns: ['dataId', 'model', 'variant', 'fidelity', 'intercept', 'C', 'nnz', 'k', 'coef_a0', 'coef_a1', 'coef_a2', 'coef_a3', 'coef_a4', 'coef_a5']


## 10 · Reload and Verify

In [19]:
attr_df = pd.read_csv(OUTPUT_DIR / f"lr_xgboost_{DATASET_ID}.csv")
print("Attribution CSV columns:", attr_df.columns.tolist())
print(f"Shape: {attr_df.shape}")
attr_df.head(3)

Attribution CSV columns: ['dataId', 'modelName', 'expMethod', 'instanceId', 'pred', 'i_max', 'a0_i', 'a1_i', 'a2_i', 'a3_i', 'a4_i', 'a5_i', 'intercept']
Shape: (80, 13)


,dataId,modelName,expMethod,instanceId,pred,i_max,a0_i,a1_i,a2_i,a3_i,a4_i,a5_i,intercept
0,wine_quality,xgboost,logistic_regression,209,0,7.065973,1.0,0.203719,-0.026374,-0.181339,-0.739809,-0.032162,-3.337674
1,wine_quality,xgboost,logistic_regression,280,0,7.144484,1.0,0.211725,-0.046952,-0.190864,-0.702855,-0.038170,-3.337674
2,wine_quality,xgboost,logistic_regression,33,0,9.028744,1.0,0.148623,-0.008256,-0.153636,-0.600035,-0.028631,-3.337674


In [20]:
table_df = pd.read_csv(OUTPUT_DIR / f"lr_xgboost_{DATASET_ID}_table.csv")
print("Surrogate table columns:", table_df.columns.tolist())
print(f"Fidelity: {table_df['fidelity'].iloc[0]:.3f}")

Surrogate table columns: ['dataId', 'model', 'variant', 'fidelity', 'intercept', 'C', 'nnz', 'k', 'coef_a0', 'coef_a1', 'coef_a2', 'coef_a3', 'coef_a4', 'coef_a5']
Fidelity: 0.863


---
## Summary

| Step | What happened |
|---|---|
| Data | Loaded `wine_quality` (400 rows, 6 features) from `assets/ai_dataset/coxam/values.csv` |
| Models | XGBoost (50 rounds) and MLP (300 epochs, batch=32) via built-in model classes |
| Predictions | `y_model = np.argmax(model.predict(X_train), axis=1)` |
| Fit surrogate | `LogisticRegressionSurrogateMethod.fit(X_train, y_model)` — C=1.0 |
| Explain | `lr.explain(X_test)` → coefficient × feature_value per instance |
| Attribution CSVs | `lr_{xgboost,mlp}_wine_quality.csv` — signed contributions |
| Surrogate tables | `lr_{xgboost,mlp}_wine_quality_table.csv` (CoXAM format with coefficients) |

**API reference**

```python
# 1. Create and fit
lr = LogisticRegressionSurrogateMethod(
    app_id="...", model_name="...", C=1.0, max_iter=1000, feature_names=[...]
)
lr.fit(X_train, np.argmax(model.predict(X_train), axis=1))

# 2. Inspect learned weights
lr.get_coefficients()   # {\"a0\": 0.42, \"a1\": -0.17, ...}
lr.get_intercept()      # -0.31

# 3. Explain test instances
result = lr.explain(X_test)

# 4. Save per-instance explanation CSV
result.save_csv(path, instance_ids, predictions, dataset_id, model_name)

# 5. Save global surrogate structure tables
lr.to_explanation_table().to_csv(table_path, index=False)
lr.to_metadata_table().to_csv(metadata_path, index=False)
```